# Enfoque híbrido

**Fase 1 (Keywords):** En esta fase se realiza la asignación de pertinencias en todos los proyectos con coincidencias de keywords registradas en el diccionario. Esta fase es exactamente igual al enfoque original de matching directo, contando solo con la adición de un registro de asignaciones como salida donde se muestra que palabras hicieron el matching en cada proyecto con 1.

**Fase 2 (Embeddings):** Es opcional y se ejecuta solo si el usuario lo indica y en el indicador que se especifique. Esta tiene el objetivo de encontrar similitudes entre los textos de los proyectos aún con 0 y las keywords registradas y mostrarlas al usuario/desarrollador con el objetivo de que este las analice y expanda el diccionario según lo considere. Esta fase como tal no realiza asignaciones, ya que se considera que, en caso contrario, se pierde el control de los criterios de asignación al depender del entrenamiento del modelo, por lo que se mantiene como apoyo para la ubicación de posibles falsos negativos y expansión del diccionario a largo plazo.


## Imports y variables globales

In [1]:
import json
import re
import numpy as np
import pandas as pd
import unicodedata

In [2]:
# Bandera para definir si el modo debug está activo
DEBUG = True

# Umbral para "cosechar" los proyectos al menos un 40% similitud con el conjunto de keywords
PRIMER_UMBRAL = 0.4

# Umbral para mostrar en terminal los fragmentos del proyecto con similitud de al menos un 50% con la keyword más similar.
SEGUNDO_UMBRAL = 0.4

## Funciones de utilidad

### Normalizar texto

In [3]:
# Función para normalizar el texto, elimina tildes, y convierte el texto a minúsculas
def normalizar(texto):
    texto = texto.lower()
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(c for c in texto if unicodedata.category(c) != "Mn")
    return texto

### Cargar modelo

In [4]:
_modelo = None

# Función para cargar modelo solo si se indica como argumento
def cargar_modelo():
    global _modelo, util
    if _modelo is None:
        print("Cargando modelo de embeddings...")
        from sentence_transformers import SentenceTransformer, util
        # Modelo preentrenado
        _modelo = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    return _modelo

### Construir texto

In [5]:
# Función para concatenar las columnas cualitativas de un proyecto en un solo texto
def construir_texto(row, columnas_texto, columnas_disponibles):
    partes = []
    for col in columnas_texto:
        if col in columnas_disponibles:
            valor = row.get(col, "")
            if isinstance(valor, str) and valor.strip():
                partes.append(f"{col}: {valor.strip()}")
    return "\n".join(partes)

### Mostrar conteos

In [6]:
# Función para mostrar los conteos
def mostrar_conteos(base, descripciones, etiqueta):
    print(f"\nConteos — {etiqueta}:")
    resultados = base[list(descripciones.keys())].sum().astype(int)
    for indicador, total in resultados.items():
        print(f"  {indicador:<70} {total}")

## Fase 1: Keywords

Busca palabras clave exactas en el texto de cada proyecto. Marca 1 donde hay coincidencias.

In [7]:
# Función para ejecutar la fase 1 de keywords
def fase_keywords(base, textos_proyectos, descripciones):
    matches_por_indicador = {ind: set() for ind in descripciones}

    print("\n--- Fase 1: Keywords ---")

    for indicador, info in descripciones.items():
        keywords = info.get("palabras_clave", [])
        if not keywords:
            continue

        if DEBUG:
            print(f"\n=== KEYWORDS {indicador} ===")

        for idx, texto in enumerate(textos_proyectos):
            texto_lower = normalizar(texto)
            coincidencias = []

            for palabra in keywords:
                patron = r"\b" + re.escape(normalizar(palabra)) + r"\b"
                if re.search(patron, texto_lower):
                    coincidencias.append(palabra)

            if coincidencias:
                matches_por_indicador[indicador].add(idx)
                base.loc[idx, indicador] = 1

                if DEBUG:
                    print(f"  [KW] Proyecto {idx + 2}: {coincidencias}")

    return matches_por_indicador

## Fase 2: Embeddings

Sugiere proyectos candidatos para enriquecer el diccionario de keywords. NO asigna valores, solo imprime en consola para revisión.

El objetivo de esta fase es dar opciones al usuario/desarrollador de posibles keywords o frases similares utilizando el JSON de descripciones_indicadores, esto con el objetivo del enriquecimiento del diccionario a la vez de evitar asignaciones abstractas de parte del modelo.

In [8]:
# Función para preparar el modelo y los embeddings de los proyectos (una sola vez)
def preparar_embeddings(textos_proyectos):
    modelo = cargar_modelo()
    total_proyectos = len(textos_proyectos)
    print(f"\nCalculando embeddings de {total_proyectos} proyectos...")
    embeddings_proyectos = modelo.encode(
        textos_proyectos, show_progress_bar=True, batch_size=64, convert_to_numpy=True,
    )
    return modelo, embeddings_proyectos


# Función para evaluar un solo indicador contra los proyectos sin match de keywords
def evaluar_indicador(indicador, info, textos_proyectos, embeddings_proyectos, matches_keywords,
                       modelo, primer_umbral=None, segundo_umbral=None):
    primer_umbral = PRIMER_UMBRAL if primer_umbral is None else primer_umbral
    segundo_umbral = SEGUNDO_UMBRAL if segundo_umbral is None else segundo_umbral

    total_proyectos = len(textos_proyectos)
    indices_todos = set(range(total_proyectos))
    sin_match = sorted(indices_todos - matches_keywords.get(indicador, set()))

    if not sin_match:
        print(f"\n=== {indicador}: todos cubiertos por keywords, skip ===")
        return

    keywords = info.get("palabras_clave", [])
    texto_indicador = " | ".join(keywords) if keywords else info.get("descripcion", indicador)
    embedding_indicador = modelo.encode(texto_indicador, convert_to_numpy=True)

    embeddings_sub = embeddings_proyectos[sin_match]
    similitudes = util.cos_sim(embedding_indicador, embeddings_sub)[0]

    print(f"\n=== EMBEDDING {indicador} (evaluando {len(sin_match)} proyectos, "
          f"umbral1={primer_umbral}, umbral2={segundo_umbral}) ===")

    encontrados = 0
    for i, idx_proyecto in enumerate(sin_match):
        score = float(similitudes[i])
        if score < primer_umbral:
            continue

        oraciones = [s.strip() for s in re.split(r'[.\n|]', textos_proyectos[idx_proyecto]) if s.strip()]
        if oraciones:
            emb_oraciones = modelo.encode(oraciones, convert_to_numpy=True)
            sims_oraciones = util.cos_sim(embedding_indicador, emb_oraciones)[0]
            mejor_idx = int(sims_oraciones.argmax())
            mejor_frag = oraciones[mejor_idx][:250]
        else:
            mejor_frag = "(sin fragmento)"

        if keywords:
            emb_kws = modelo.encode(keywords, convert_to_numpy=True)
            emb_proyecto = embeddings_proyectos[idx_proyecto]
            sims_kws = util.cos_sim(emb_proyecto, emb_kws)[0]
            mejor_kw_idx = int(sims_kws.argmax())
            mejor_kw = keywords[mejor_kw_idx]
            mejor_kw_score = float(sims_kws[mejor_kw_idx])
        else:
            mejor_kw, mejor_kw_score = "(sin keywords)", 0.0

        if mejor_kw_score > segundo_umbral:
            encontrados += 1
            print(
                f"  [EMB] Proyecto {idx_proyecto + 2}: score={score:.3f}\n"
                f"        Fragmento : {mejor_frag}\n"
                f"        Similar a : '{mejor_kw}' ({mejor_kw_score:.3f})\n"
            )

    if encontrados == 0:
        print("  (sin candidatos que superen ambos umbrales)")

## Procesamiento

### Carga de datos

In [9]:
ruta_entrada = "../../info/df_combinado_limpio.xlsx"

base = pd.read_excel(ruta_entrada)

with open("columnas.json", encoding="utf-8") as f:
    columnas = json.load(f)

with open("descripciones_indicadores.json", encoding="utf-8") as f:
    descripciones = json.load(f)

descripciones.pop("inactivos", None)

for indicador in descripciones:
    base[indicador] = 0

columnas_disponibles = set(base.columns)
textos_proyectos = base.apply(
    construir_texto, axis=1,
    columnas_texto=columnas["columnas_texto"],
    columnas_disponibles=columnas_disponibles,
).tolist()

print(f"{len(textos_proyectos)} proyectos cargados.")

578 proyectos cargados.


### Fase 1

In [10]:
matches_keywords = fase_keywords(base, textos_proyectos, descripciones)
mostrar_conteos(base, descripciones, "después de keywords")


--- Fase 1: Keywords ---

=== KEYWORDS A.1.1 Agua ===
  [KW] Proyecto 72: ['agua', 'rios']
  [KW] Proyecto 94: ['agua', 'acuatico']
  [KW] Proyecto 107: ['rio']
  [KW] Proyecto 109: ['agua', 'recurso hídrico', 'potable']
  [KW] Proyecto 120: ['agua']
  [KW] Proyecto 161: ['agua']
  [KW] Proyecto 170: ['agua', 'aguas']
  [KW] Proyecto 190: ['agua']
  [KW] Proyecto 194: ['agua']
  [KW] Proyecto 216: ['agua']
  [KW] Proyecto 222: ['agua']
  [KW] Proyecto 225: ['agua', 'aguas', 'recurso hídrico', 'potable']
  [KW] Proyecto 234: ['agua', 'recurso hídrico', 'potable', 'manantiales']
  [KW] Proyecto 266: ['recurso hídrico', 'cuencas']
  [KW] Proyecto 275: ['agua']
  [KW] Proyecto 296: ['agua']
  [KW] Proyecto 318: ['agua', 'acueductos', 'alcantarillados']
  [KW] Proyecto 327: ['rio']
  [KW] Proyecto 328: ['acuatica']
  [KW] Proyecto 356: ['agua', 'recurso hídrico']
  [KW] Proyecto 377: ['agua', 'potable']
  [KW] Proyecto 408: ['agua']
  [KW] Proyecto 443: ['acueductos', 'alcantarillados']
  

### Preparar embeddings

In [11]:
modelo, embeddings_proyectos = preparar_embeddings(textos_proyectos)

Cargando modelo de embeddings...


/mnt/c/Users/Aron/Desktop/U/Cursos/2026/Asistencia/code/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 511.97it/s]



Calculando embeddings de 578 proyectos...


Batches: 100%|██████████| 10/10 [00:25<00:00,  2.52s/it]


### Celda de trabajo

In [46]:
with open("descripciones_indicadores.json", encoding="utf-8") as f:
    descripciones_actualizado = json.load(f)
descripciones_actualizado.pop("inactivos", None)

indicador_actual = "A.4.4 Agropecuario"

evaluar_indicador(
    indicador_actual, descripciones_actualizado[indicador_actual],
    textos_proyectos, embeddings_proyectos, matches_keywords, modelo
)


=== EMBEDDING A.4.4 Agropecuario (evaluando 578 proyectos, umbral1=0.4, umbral2=0.4) ===
  [EMB] Proyecto 31: score=0.631
        Fragmento : Agricultura
        Similar a : 'agropecuario' (0.597)

  [EMB] Proyecto 66: score=0.550
        Fragmento : Objetivos específicos: 1: Realizar un diagnóstico sobre la producción de maíz pujagua en cuatro cantones de Guanacaste (Nicoya, Santa Cruz, Carrillo y La Cruz) para identificar los factores sociales, ambientales, económicos y agronómicos que inciden 
        Similar a : 'agroalimentario' (0.516)

  [EMB] Proyecto 129: score=0.598
        Fragmento : Descriptores: Agricultura
        Similar a : 'agropecuario' (0.476)

  [EMB] Proyecto 130: score=0.643
        Fragmento : Agricultura
        Similar a : 'agricultura' (0.550)

  [EMB] Proyecto 145: score=0.604
        Fragmento : Descriptores: Agricultura
        Similar a : 'agroalimentario' (0.510)

  [EMB] Proyecto 151: score=0.642
        Fragmento : Agricultura
        Similar a : 'agr

In [44]:
with open("descripciones_indicadores.json", encoding="utf-8") as f:
    descripciones = json.load(f)
descripciones.pop("inactivos", None)

indicador_a_mostrar = "A.4.4 Agropecuario"
base[indicador_a_mostrar] = 0

matches_keywords_parcial = fase_keywords(base, textos_proyectos, {indicador_a_mostrar: descripciones[indicador_a_mostrar]})
mostrar_conteos(base, {indicador_a_mostrar: descripciones[indicador_a_mostrar]}, "después de keywords (actualizado)")


--- Fase 1: Keywords ---

=== KEYWORDS A.4.4 Agropecuario ===
  [KW] Proyecto 31: ['agricultura']
  [KW] Proyecto 44: ['agricultura']
  [KW] Proyecto 66: ['agricultura']
  [KW] Proyecto 128: ['agricultura']
  [KW] Proyecto 129: ['agricultura', 'agro']
  [KW] Proyecto 130: ['agricultura']
  [KW] Proyecto 145: ['agricultura']
  [KW] Proyecto 151: ['agricultura', 'agroalimentario']
  [KW] Proyecto 152: ['agroalimentario']
  [KW] Proyecto 153: ['agropecuario', 'agroalimentario', 'agro']
  [KW] Proyecto 155: ['agricultura']
  [KW] Proyecto 166: ['agroalimentario']
  [KW] Proyecto 190: ['agricultura']
  [KW] Proyecto 200: ['agricultura', 'agro']
  [KW] Proyecto 204: ['agropecuario', 'agricultura', 'agroalimentario']
  [KW] Proyecto 217: ['agroalimentario']
  [KW] Proyecto 226: ['agroalimentario']
  [KW] Proyecto 227: ['agroalimentario']
  [KW] Proyecto 251: ['agricultura']
  [KW] Proyecto 270: ['agroalimentario']
  [KW] Proyecto 318: ['agricultura']
  [KW] Proyecto 335: ['agricultura']
  [K